# Agent 01 Trades Ticks Download

Este notebook lanza el descargador de `trades_ticks` con enfoque de producci?n:

- `resume` por `task_key`
- `retries` con `backoff` para errores transitorios
- m?tricas por batch
- pol?tica de concurrencia expl?cita


In [3]:
from pathlib import Path

RUN_ID = 'trades_ticks_prod_2005_2026'
RUN_DIR = Path(r'C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit') / RUN_ID
OUTPUT_ROOT = Path(r'D:\trades_ticks_prod_2005_2026')
TASKS_CSV = RUN_DIR / 'inputs' / 'tasks_trades_ticks.csv'
SCRIPT = Path(r'C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\cell_code\00_data_certification\201_agent1_download_trades_ticks_realtime.py')

SESSION = 'market'
CONCURRENT = 8
TASK_BATCH_SIZE = 200
MAX_RETRIES = 4
RETRY_BASE_SLEEP_SEC = 0.75
RETRY_MAX_SLEEP_SEC = 12.0
RESUME = True

print('RUN_DIR:', RUN_DIR)
print('OUTPUT_ROOT:', OUTPUT_ROOT)
print('TASKS_CSV exists:', TASKS_CSV.exists())
print('SCRIPT exists:', SCRIPT.exists())

RUN_DIR: C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\trades_ticks_prod_2005_2026
OUTPUT_ROOT: D:\trades_ticks_prod_2005_2026
TASKS_CSV exists: False
SCRIPT exists: True


## Pol?tica Operativa

Recomendaci?n actual de concurrencia:

- `<= 8`: `stable_default`
- `9-12`: `fast_balanced`
- `> 12`: `aggressive_monitor_429`

Para producci?n, empezar por `CONCURRENT = 8` y subir solo si el ratio de `DOWNLOAD_FAIL` y `429` sigue bajo.


In [6]:
from pathlib import Path

RUN_ID = "trades_ticks_prod_2005_2026"
RUN_DIR = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit") / RUN_ID
SCRIPT = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\cell_code\00_data_certification\205_generate_trades_ticks_tasks.py")

cmd = f'''python {SCRIPT} --run-id {RUN_ID} --run-dir {RUN_DIR} --mode lifecycle --cutoff 2026-03-14'''
print(cmd)

python C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\cell_code\00_data_certification\205_generate_trades_ticks_tasks.py --run-id trades_ticks_prod_2005_2026 --run-dir C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\trades_ticks_prod_2005_2026 --mode lifecycle --cutoff 2026-03-14


Qué hará:

  - creará:
      - C:
        \TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\trades_ticks_prod_2005_2026\inputs\tasks_trades_tic
        ks.csv
  - y también:
      - tickers_trades_ticks.csv
      - tasks_trades_ticks_meta.json

```bash
python C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\cell_code\00_data_certification\205_generate_trades_ticks_tasks.py --run-id trades_ticks_prod_2005_2026 --run-dir C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\trades_ticks_prod_2005_2026 --mode lifecycle --cutoff 2026-03-14

C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\trades_ticks_prod_2005_2026\inputs\tasks_trades_ticks.csv
C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\trades_ticks_prod_2005_2026\inputs\tickers_trades_ticks.csv
C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\trades_ticks_prod_2005_2026\inputs\tasks_trades_ticks_meta.json
{
  "mode": "lifecycle",
  "universe_path": "C:\\TSIS_Data\\v1\\backtest_SmallCaps\\data\\reference\\universe_pti\\tickers_2005_2026_upper_excluding_ohlcv_1m_missing_vs_daily_ge_2B_or_null.parquet",
  "lifecycle_path": "C:\\TSIS_Data\\v1\\backtest_SmallCaps\\data\\reference\\official_lifecycle_compiled.csv",
  "tickers_count": 1962,
  "tasks_total": 3068656,
  "date_min": "1995-09-13",
  "date_max": "2026-03-13",
  "run_id": "trades_ticks_prod_2005_2026"
```

 Si tasks_trades_ticks.csv sale True, ya pasamos a Agent01.

In [9]:
from pathlib import Path

RUN_ID = "trades_ticks_prod_2005_2026"
RUN_DIR = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit") / RUN_ID

for name in ["tasks_trades_ticks.csv", "tickers_trades_ticks.csv", "tasks_trades_ticks_meta.json"]:
    p = RUN_DIR / "inputs" / name
    print(name, p.exists(), p)

tasks_trades_ticks.csv True C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\trades_ticks_prod_2005_2026\inputs\tasks_trades_ticks.csv
tickers_trades_ticks.csv True C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\trades_ticks_prod_2005_2026\inputs\tickers_trades_ticks.csv
tasks_trades_ticks_meta.json True C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\trades_ticks_prod_2005_2026\inputs\tasks_trades_ticks_meta.json


**Ahora sí puedes lanzar Agent01 con este comando:**

```bash
PS C:\Users\AlexJ> python C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\cell_code\00_data_certification\201_agent1_download_trades_ticks_realtime.py --csv C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\trades_ticks_prod_2005_2026\inputs\tasks_trades_ticks.csv --output D:\trades_ticks_prod_2005_2026 --run-id trades_ticks_prod_2005_2026 --run-dir C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\trades_ticks_prod_2005_2026 --session market --concurrent 8 --task-batch-size 200 --max-retries 4 --retry-base-sleep-sec 0.75 --retry-max-sleep-sec 12.0 --resume

run_id=trades_ticks_prod_2005_2026
run_dir=C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\trades_ticks_prod_2005_2026
output_root=D:\trades_ticks_prod_2005_2026
csv=C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\trades_ticks_prod_2005_2026\inputs\tasks_trades_ticks.csv
session=market
tasks_total=3068656
tasks_already_ok=0
tasks_to_process=3068656
concurrency_policy=stable_default
processed=25/3068656
processed=50/3068656
processed=75/3068656
processed=100/3068656
processed=125/3068656
processed=150/3068656
processed=175/3068656
processed=200/3068656
batch_done=200/3068656 | ok=190 empty=10 fail=0 | rows=7968811 elapsed_sec=667.79 | tasks_per_sec=0.30
processed=225/3068656
processed=250/3068656
processed=275/3068656
processed=300/3068656
processed=325/3068656
processed=350/3068656
```

## Visor de Estado y M?tricas de Batch

Agent01 escribe:

- `download_events_trades_ticks_current.csv`
- `download_events_trades_ticks_history.csv`
- `live_status_trades_ticks_download.json`
- `expected_manifest_trades_ticks.csv`

El `live_status` ahora incluye `batch_metrics` para throughput y estabilidad.


In [10]:
from pathlib import Path
import json
import pandas as pd

live = RUN_DIR / 'live_status_trades_ticks_download.json'
cur = RUN_DIR / 'download_events_trades_ticks_current.csv'
hist = RUN_DIR / 'download_events_trades_ticks_history.csv'
manifest = RUN_DIR / 'expected_manifest_trades_ticks.csv'
run_cfg = RUN_DIR / 'run_config_trades_ticks_download.json'

for p in [live, cur, hist, manifest, run_cfg]:
    print(p.name, p.exists())

if run_cfg.exists():
    print('\nrun_config')
    print(json.dumps(json.loads(run_cfg.read_text(encoding='utf-8')), indent=2, ensure_ascii=False))

if live.exists():
    print('\nlive_status')
    print(json.dumps(json.loads(live.read_text(encoding='utf-8')), indent=2, ensure_ascii=False))

if cur.exists():
    df = pd.read_csv(cur)
    display(df.head(20))
    if 'status' in df.columns:
        print('\nstatus_counts')
        display(df['status'].value_counts(dropna=False).rename_axis('status').reset_index(name='tasks'))

if manifest.exists():
    mdf = pd.read_csv(manifest)
    print('\nexpected_manifest rows:', len(mdf))
    display(mdf.head(20))


live_status_trades_ticks_download.json False
download_events_trades_ticks_current.csv False
download_events_trades_ticks_history.csv False
expected_manifest_trades_ticks.csv True
run_config_trades_ticks_download.json True

run_config
{
  "run_id": "trades_ticks_prod_2005_2026",
  "run_dir": "C:\\TSIS_Data\\v1\\backtest_SmallCaps\\runs\\polygon_realtime_audit\\trades_ticks_prod_2005_2026",
  "csv_path": "C:\\TSIS_Data\\v1\\backtest_SmallCaps\\runs\\polygon_realtime_audit\\trades_ticks_prod_2005_2026\\inputs\\tasks_trades_ticks.csv",
  "output_root": "D:\\trades_ticks_prod_2005_2026",
  "session": "market",
  "concurrent": 8,
  "task_batch_size": 200,
  "resume": true,
  "max_retries": 4,
  "retry_base_sleep_sec": 0.75,
  "retry_max_sleep_sec": 12.0,
  "recommended_concurrency_policy": "stable_default"
}

expected_manifest rows: 3068656


,task_key,ticker,date,session,expected_file
0,AABA|2017-06-16|market,AABA,2017-06-16,market,D:\trades_ticks_prod_2005_2026\AABA\year=2017\...
1,AABA|2017-06-19|market,AABA,2017-06-19,market,D:\trades_ticks_prod_2005_2026\AABA\year=2017\...
2,AABA|2017-06-20|market,AABA,2017-06-20,market,D:\trades_ticks_prod_2005_2026\AABA\year=2017\...
3,AABA|2017-06-21|market,AABA,2017-06-21,market,D:\trades_ticks_prod_2005_2026\AABA\year=2017\...
4,AABA|2017-06-22|market,AABA,2017-06-22,market,D:\trades_ticks_prod_2005_2026\AABA\year=2017\...
5,AABA|2017-06-23|market,AABA,2017-06-23,market,D:\trades_ticks_prod_2005_2026\AABA\year=2017\...
6,AABA|2017-06-26|market,AABA,2017-06-26,market,D:\trades_ticks_prod_2005_2026\AABA\year=2017\...
7,AABA|2017-06-27|market,AABA,2017-06-27,market,D:\trades_ticks_prod_2005_2026\AABA\year=2017\...
8,AABA|2017-06-28|market,AABA,2017-06-28,market,D:\trades_ticks_prod_2005_2026\AABA\year=2017\...
9,AABA|2017-06-29|market,AABA,2017-06-29,market,D:\trades_ticks_prod_2005_2026\AABA\year=2017\...


## Lectura R?pida

- `DOWNLOADED_OK`: file materializado
- `DOWNLOADED_EMPTY`: llamada correcta, pero sin trades para esa sesi?n
- `DOWNLOAD_FAIL`: error real de descarga

Las m?tricas de batch que importan para tuning:

- `batch_fail`
- `batch_elapsed_sec`
- `batch_tasks_per_sec`
- `batch_mean_task_sec`

Si `batch_fail` o `429` suben, bajar `CONCURRENT`.
